## Data Quality Assessment and Cleaning

### Purpose

This notebook creates a reliable analytical foundation by assessing and improving the quality of the raw DataCo supply chain data. Given the dataset's operational and financial complexity, careful data preparation is important to ensure that downstream forecasting, risk modeling, and BI reporting are defensible and built on verifiably clean data.

### What This Notebook Does

- **Structural cleanup:** Remove fully empty columns and sparsely populated fields that offer limited analytical value.
- **Schema standardization:** Rename columns to snake_case, parse date fields, trim whitespace from strings, and enforce correct data types.
- **Deduplication:** Identify and remove duplicate records at the order-item grain to protect demand, revenue, and service-level metric accuracy.
- **Feature derivation:** Compute operational metrics (actual delivery date, shipping delay, processing time) and a profit margin proxy for downstream analysis.
- **Outlier detection:** Flag extreme values across financial, delivery, pricing, and quantity fields using the IQR method to separate genuine business events from potential data errors.
- **Data governance check:** Validate that discount rates do not exceed 100%, confirming transactional logic integrity.
- **Data horizon confirmation:** Verify the full timeline of supply chain operations by identifying the earliest and latest order dates.

### Output

An analytics-ready silver dataset that serves as a unified foundation for subsequent notebooks and Power BI dashboards.

## Load DataCo Dataset

In [1]:
import numpy as np
import pandas as pd

# Set display options for dataframe
pd.set_option("display.max_columns", 60)

# Load DataCo dataset
dataco_raw_path = "/kaggle/input/dataco-smart-supply-chain-for-big-data-analysis/DataCoSupplyChainDataset.csv"
dataco_raw = pd.read_csv(dataco_raw_path, encoding = "latin-1", low_memory = False)

dataco_raw.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,1/13/2018 11:45,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,1/13/2018 11:24,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


## Missingness Analysis

In [2]:
dataco_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

In [3]:
# Isolate raw data to preserve original ingestion state
dataco_raw1 = dataco_raw.copy()

# Drop columns with zero populated values
dataco_raw1.dropna(axis = 1, how = 'all', inplace = True)

# Drop Zipcode: ~86% missing, City/State columns provide sufficient geographic granularity
if 'Order Zipcode' in dataco_raw1.columns:
    dataco_raw1.drop(columns = ['Order Zipcode'], inplace = True)

In [4]:
dataco_raw1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 51 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

#### Missingness Findings and Treatment Decisions

**Fully Missing Column:** `Product Description`

The `Product Description` field is **100% null across all records**, confirming that no usable information is present. The column is therefore dropped from the dataset.

From a data governance and SCOR Enable perspective, analytical datasets should be intentionally minimal. Retaining a fully empty attribute adds structural noise without providing analytical value, complicates schema maintenance, and increases cognitive load for downstream users. Removing this column improves clarity and supports a cleaner, more purposeful analytical model.

**Highly Sparse Column:** `order_zipcode`

The `Order Zipcode` field contains **24,840 non-null values** in a dataset of approximately 180,000 records, indicating that **roughly 86% of values are missing**.

This pattern suggests **systematic, rather than random missingness**. In practice, postal codes are often captured only for specific geographies (e.g., domestic U.S. orders) or customer segments, while international or third-party shipments may omit this level of granularity.

Using this field directly in geographic or service-level analyses (e.g., late delivery heatmaps) would introduce significant selection bias, as insights would reflect only a narrow and non-representative subset of the supply chain.

The column is therefore excluded from the core analytical model, and instead `Order city`, `Order state`, and `Order region` will be relied upon since these columns are fully populated and provide consistent geographic coverage across all orders.


Prioritizing complete and consistently captured location attributes ensures that logistics performance, risk modeling, and customer service analyses reflect the entire supply chain footprint instead of a geographically biased slice.

## Data Cleansing and Deduplication

In [5]:
# Standardise names - strip parentheses and convert to snake_case
dataco_refined = dataco_raw1.rename(columns = {
    "order date (DateOrders)": "order_date",
    "shipping date (DateOrders)": "shipping_date",
    "Late_delivery_risk": "late_delivery_risk"
})

# Lowercase all remaining columns, remove parentheses, and replace spaces with underscores
dataco_refined.columns = (dataco_refined.columns
                      .str.replace(r'[\(\)]', '', regex = True) 
                      .str.replace(' ', '_')                 
                      .str.lower())

In [6]:
# Parse date columns 
date_columns = ["order_date", "shipping_date"]
for column in date_columns:
    if column in dataco_refined.columns:
        dataco_refined[column] = pd.to_datetime(dataco_refined[column], errors = "coerce")

# Trim strings
obj_columns = dataco_refined.select_dtypes(include = "object").columns
for column in obj_columns:
    dataco_refined[column] = dataco_refined[column].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Standardize Target - SCOR Reliability Flag
target = "late_delivery_risk"
if target in dataco_refined.columns:
    dataco_refined[target] = pd.to_numeric(dataco_refined[target], errors = "coerce").fillna(0).astype(int)

# Remove duplicates for accurate Perfect Order Fulfillment metrics
key_columns = ["order_id", "product_card_id", "customer_id", "order_date"]
existing_keys = [column for column in key_columns if column in dataco_refined.columns]

if existing_keys:
    duplicate_rate = dataco_refined.duplicated(subset = existing_keys).mean()
    print(f"Duplicate Rate at Order-Item Grain: {duplicate_rate:.2%}")
    dataco_refined = dataco_refined.drop_duplicates(subset = existing_keys)

print("Final shape after refinement:", dataco_refined.shape)

Duplicate Rate at Order-Item Grain: 11.50%
Final shape after refinement: (159763, 51)


#### Duplicate Records Analysis and Treatment

The observed **11.5% duplicate rate** at the order-item grain reflects the way transactional supply chain data is stored, exported, and flattened for analytical use. In the DataCo Smart Supply Chain dataset, duplication is not inherently an error; however, distinguishing legitimate multi-line transactions from true data inflation is important for accurate performance and risk analysis.

##### Expected Duplication at the Order-Item Grain

The most common source of apparent duplication arises from the dataset’s order-item granularity.

A single `order_id` can legitimately appear across multiple rows because each row represents a distinct product within the same customer order. For example, if a customer purchases both footwear and apparel in a single transaction, the dataset records one row per `product_card_id`, all sharing the same `order_id`.

At a high level, these rows may appear identical across fields such as order date, customer attributes, or shipping method. However, they represent unique physical items, rather than duplicate transactions.

This structure is analytically valid and must be preserved to support product-level margin analysis, item-level demand forecasting, and SKU-specific risk modeling.

##### Structural Duplication from Relational Join Inflation

Beyond legitimate order-item expansion, additional duplication can be introduced through relational join inflation.

The dataset is a flattened export combining multiple operational domains (e.g., sales, provisioning, production, and distribution). When one-to-many relationships exist - such as an order linked to multiple status updates or logistics events - flattening these tables can result in repeated representations of the same order-item combination.

While technically accurate at the system level, these repetitions distort analytical measures if not addressed.

##### Operational Data Artifacts

In operational supply chain and ERP environments, systematic duplication can also occur due to:

- System synchronization effects, where regional systems are merged without a perfectly unified global identifier.
- Order lifecycle updates, where changes in status (e.g., Processing → Shipped → Complete) are stored as separate records but appear as duplicates in a flat analytical extract.

These artifacts are common in operational data and must be normalized for analytical use.

The reason deduplication matters form a SCOR perspective is that without controlled deduplication, key SCOR-aligned metrics become significantly biased:

- **Reliability and Risk Modeling:**
A single late shipment containing multiple items may be misinterpreted as multiple independent delivery failures, inflating late-delivery risk estimates.
- **Financial Performance Metrics:**
Sales, revenue, and profit measures can be artificially double-counted by repeated data rows, overstating true business performance.

Such distortions undermine both predictive models and executive-level KPIs.

The deduplication Strategy retains one record per unique product within each order, removes technical and relational duplicates, and preserves the correct grain for product-level, financial, and risk analyses.

After refinement, the dataset contains **159,763 records across 51 features**, representing a clean and analytically consistent view of transactional supply chain activity.

## Derived Operational and Financial Features

To support SCOR-aligned performance measurement and downstream modeling, several derived features are constructed to translate raw transactional fields into meaningful operational and financial indicators. These features capture key aspects of order execution, service reliability, and profitability that are not represented in the source data but are important for forecasting, risk analysis, and KPI reporting.

Specifically:

- **Delivery completion metrics** are derived to establish a realistic view of order fulfillment timelines, enabling service-level and late-delivery analyses.
- **Delay and processing measures** decompose total order cycle time into pre-shipment and transit components, improving interpretability of operational bottlenecks.
- **Profitability proxies** normalize financial outcomes at the order level, allowing consistent comparison across customers, products, and regions.

All derived fields are calculated using deterministic, auditable transformations to ensure transparency and reproducibility across notebooks and BI layers.

In [ ]:
# Actual delivery date - when the order reached the customer
dataco_refined['actual_delivery_date'] = (
    dataco_refined['shipping_date'] + 
    pd.to_timedelta(dataco_refined['days_for_shipping_real'], unit = 'D')
)

# Shipping delay - variance between scheduled and actual transit days
dataco_refined["shipping_delay"] = dataco_refined["days_for_shipping_real"] - dataco_refined["days_for_shipment_scheduled"]

# Order processing time - days between order placement and carrier handoff
dataco_refined["processing_days"] = (dataco_refined['shipping_date'] - dataco_refined['order_date']).dt.days

# Profit margin - derived ratio for financial performance benchmarking
dataco_refined["profit_margin"] = np.where(
    dataco_refined['sales_per_customer'].astype(float) != 0,
    dataco_refined['order_profit_per_order'].astype(float) / dataco_refined['sales_per_customer'].astype(float),
    np.nan
)

## Outlier Assessment Across Financial and Delivery Metrics

In [8]:
dataco_refined.describe()

,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,late_delivery_risk,category_id,customer_id,customer_zipcode,department_id,latitude,longitude,order_customer_id,order_date,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,product_card_id,product_category_id,product_price,product_status,shipping_date,actual_delivery_date,shipping_delay,processing_days,profit_margin
count,159763.000000,159763.000000,159763.000000,159763.000000,159763.000000,159763.000000,159763.000000,159760.000000,159763.000000,159763.000000,159763.000000,159763.000000,159763,159763.000000,159763.000000,159763.000000,159763.000000,159763.000000,159763.000000,159763.000000,159763.000000,159763.000000,159763.000000,159763.000000,159763.000000,159763.000000,159763.000000,159763.0,159763,159763,159763.000000,159763.000000,159763.000000
mean,3.499690,2.932356,21.778624,181.846755,0.548838,32.170071,6757.191947,35899.981378,5.461709,29.717599,-84.902957,6757.191947,2016-06-16 02:12:49.258714368,36451.466209,698.647709,20.419859,0.101100,90796.577825,142.146061,0.120377,2.091266,202.266365,181.846755,21.778624,698.647709,32.170071,142.146061,0.0,2016-06-19 14:14:02.807658752,2016-06-23 02:13:36.038006016,0.567334,3.473914,0.120180
min,0.000000,0.000000,-4274.979980,7.490000,0.000000,2.000000,1.000000,603.000000,2.000000,-33.937553,-158.025986,1.000000,2015-01-01 00:00:00,1.000000,19.000000,0.000000,0.000000,1.000000,9.990000,-2.750000,1.000000,9.990000,7.490000,-4274.979980,19.000000,2.000000,9.990000,0.0,2015-01-03 00:00:00,2015-01-03 15:51:00,-2.000000,0.000000,-2.750813
25%,2.000000,2.000000,6.750000,100.779999,0.000000,18.000000,3286.000000,725.000000,4.000000,18.265404,-98.443069,3286.000000,2015-09-23 06:48:00,18174.000000,403.000000,5.200000,0.040000,45424.000000,50.000000,0.080000,1.000000,119.980003,100.779999,6.750000,403.000000,18.000000,50.000000,0.0,2015-09-26 19:57:00,2015-09-30 08:02:00,0.000000,2.000000,0.074967
50%,3.000000,4.000000,30.740000,161.970001,1.000000,29.000000,6505.000000,19149.000000,5.000000,33.144863,-76.777069,6505.000000,2016-06-14 23:46:00,36376.000000,627.000000,13.750000,0.090000,90821.000000,70.000000,0.270000,1.000000,199.919998,161.970001,30.740000,627.000000,29.000000,70.000000,0.0,2016-06-18 11:10:00,2016-06-22 02:04:00,1.000000,3.000000,0.270013
75%,5.000000,4.000000,64.019997,247.399994,1.000000,45.000000,9847.500000,78201.000000,7.000000,39.279617,-66.370583,9847.500000,2017-03-06 08:10:00,54485.000000,1004.000000,29.500000,0.160000,136265.500000,199.990005,0.360000,3.000000,299.950012,247.399994,64.019997,1004.000000,45.000000,199.990005,0.0,2017-03-10 02:28:00,2017-03-13 15:00:00,1.000000,5.000000,0.362998
max,6.000000,4.000000,911.799988,1939.989990,1.000000,76.000000,20757.000000,99205.000000,12.000000,48.781933,115.263077,20757.000000,2018-01-31 23:38:00,77204.000000,1363.000000,500.000000,0.250000,180519.000000,1999.989990,0.500000,5.000000,1999.989990,1939.989990,911.799988,1363.000000,76.000000,1999.989990,0.0,2018-02-06 22:14:00,2018-02-12 22:14:00,4.000000,6.000000,0.500611
std,1.624563,1.374203,105.014993,122.920852,0.497611,15.869029,4226.595036,37538.054532,1.652639,9.816072,21.441820,4226.595036,NaN,21184.413928,339.853612,22.021058,0.070262,52400.561554,142.624210,0.467921,1.433277,135.515254,122.920852,105.014993,339.853612,15.869029,142.624210,0.0,NaN,NaN,1.492336,1.671281,0.467785


In [9]:
# Function to identify outliers
def detect_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR
    return df[(df[column] < lower_fence) | (df[column] > upper_fence)]

# Delivery outliers 
outlier_delays = detect_outliers(dataco_refined, "shipping_delay")
print(f"Number of shipping delay outliers: {len(outlier_delays)}")

# Financial outliers 
profit_per_order_outliers = detect_outliers(dataco_refined, 'order_profit_per_order')
print(f"Profit per Order Outliers: {len(profit_per_order_outliers)}")

sales_per_customer_outliers = detect_outliers(dataco_refined, 'sales_per_customer')
print(f"Sales per Customer Outliers: {len(sales_per_customer_outliers)}")

# Pricing and discount outliers 
order_item_discount_outliers = detect_outliers(dataco_refined, 'order_item_discount')
print(f"Discount Amount Outliers: {len(order_item_discount_outliers)}")

order_item_discount_rate_outliers = detect_outliers(dataco_refined, 'order_item_discount_rate')
print(f"Discount Rate Outliers: {len(order_item_discount_rate_outliers)}")

# Quantity outliers 
order_item_quantity_outliers = detect_outliers(dataco_refined, 'order_item_quantity')
print(f"Quantity Outliers: {len(order_item_quantity_outliers)}")

Number of shipping delay outliers: 31656
Profit per Order Outliers: 16978
Sales per Customer Outliers: 1639
Discount Amount Outliers: 6660
Discount Rate Outliers: 0
Quantity Outliers: 0


In [10]:
# Data governance check: Validate discount consistency
invalid_discounts = dataco_refined[dataco_refined['order_item_discount_rate'] > 1.0]

if invalid_discounts.empty:
    print("DATA INTEGRITY IS VERIFIED: No discount rates exceed 100%. Financial transactional logic is sound.")
else:
    print(f"DATA QUALITY ALERT: {len(invalid_discounts)} Rows found with invalid discount rates (>100%).")
    display(invalid_discounts.head())

DATA INTEGRITY IS VERIFIED: No discount rates exceed 100%. Financial transactional logic is sound.


#### Outlier Analysis and Business Interpretation

Outliers were identified using the Interquartile Range (IQR) method, a robust, distribution-agnostic approach well-suited to operational and financial data. Importantly, outlier detection in this context is treated as a **diagnostic step**, rather than an automatic trigger for data removal. Each result is evaluated through a supply chain and SCOR lens to distinguish legitimate data quality issues from meaningful business signals.

##### **1. Shipping Delay Outliers (31,656 records)**

Shipping delay shows the highest outlier count, making it the most operationally significant finding in this analysis.

From the descriptive statistics:

- **Q1:** 0 days (on-time delivery)
- **Median:** 1 day late
- **Q3:** 1 day late
- **Minimum:** -2 days (early delivery)
- **Maximum:** 4 days late

This produces a very tight IQR:

- **IQR:** 1 day
- **Lower fence:** -1.5 days
- **Upper fence:** 2.5 days

As a result:

- Orders delayed by 3-4 days are flagged as upper outliers.
- Orders delivered 2 days early are flagged as lower outliers.

The high outlier count (~17% of orders) does not indicate extreme operational failure. Instead, it reveals that the statistical norm of this supply chain is tightly centered around on-time to one day late delivery. Deviations beyond a two-day window are statistically infrequent, even if, from a customer perspective, they may represent inconvenience rather than severe disruption.

This finding highlights a **systemic reliability challenge** (operational signal), rather than a data error. **All shipping delay observations are therefore kept in full**, as these are central to modeling `late_delivery_risk` and understanding service-level variability within the SCOR Deliver process.

##### **2. Financial Outliers: Profit per Order (16,978 records)**

Profit per order shows a pronounced long tail, driven by both deep losses and exceptionally high-margin transactions.

Negative profit values are not errors; they are core to the SCOR Return process and likely reflect returns, heavy discounting, expedited shipping costs, or loss-leading strategies - where certain products are priced below cost to drive customer acquisition or larger basket sizes. Conversely, high-margin outliers may reflect differentiated products, full-price sales, or premium pricing on scarce items.

Thus, **all profit outliers are retaind**. Removing these values would artificially smooth profitability metrics, masking value at risk and undermining financial performance analysis. These observations are important for identifying loss drivers, margin volatility, and improvement opportunities.

##### **3. Revenue Outliers: Sales per Customer (1,639 records)**

Sales per customer shows a smaller but meaningful number of outliers. These are likely high-volume or high-value orders, often associated with enterprise, wholesale, or B2B customers.

These observations are **retained in full** since these customers disproportionately influence revenue concentration and are importanat for downstream analyses such as ABC classification, customer segmentation, and strategic account management. Excluding them would distort demand and profitability insights.

##### **4. Discount Metrics**

- **Discount Rate (`order_item_discount_rate`):** 0 outliers   
This indicates strong pricing discipline and validates the integrity of discount rate data. No corrective action is required.

- **Discount Amount (`order_item_discount`):** 6,660 outliers   
These represent extreme absolute discounts, often tied to high-value orders rather than erroneous rates.

These **observations are kept in full**, though discount amount consistency is validated against discount rate to detect true data entry errors (e.g., rates exceeding 100%). 

These observations are valuable for assessing pricing agility and promotional impact within the SCOR Plan and Deliver processes.

##### **5.  Order Quantity Outliers (0 records)**

Order item quantity shows no detected outliers.

From the distribution:

- **Q1:** 1
- **Q3:** 3
- **IQR:** 2
- **Upper fence:** 6
- **Maximum observed value:** 5

Because observed quantities remain within operational bounds, the IQR method correctly flags no extremes. This reflects a highly standardized fulfillment process, likely enforced through warehouse or system constraints and serves as a positive indicator of process control in the SCOR Source and Make phases.

##### **Summary Decision Framework**

Across all variables, outliers are retained by default unless there is clear evidence of data corruption. In this dataset, most extreme values represent meaningful operational or financial behavior, rather than noise. Keeping all observations ensures that forecasting models, risk estimates, and executive KPIs reflect the true **variability and stress points** of the supply chain.

## Data Horizon

In [11]:
# Check the full timeline of the supply chain operations
print(f"Data Horizon: {dataco_refined['order_date'].min()} to {dataco_refined['order_date'].max()}")

Data Horizon: 2015-01-01 00:00:00 to 2018-01-31 23:38:00


## Persisting Silver Dataset

All important data quality checks, refinements, and feature derivations have been completed. The resulting dataset represents a curated, analytics-ready silver layer, providing a stable and consistent foundation for downstream modeling, forecasting, and BI reporting.

Persisting this dataset formalizes the separation between raw ingestion and analytical usage, improving reproducibility, traceability, and performance across subsequent notebooks and Power BI dashboards.

In [12]:
# Save to Parquet 
file_name = "orders_silver.parquet"
dataco_refined.to_parquet(file_name, index = False)

print(f"File saved successfully: {file_name}")

dataco_refined.info()

File saved successfully: orders_silver.parquet
<class 'pandas.core.frame.DataFrame'>
Index: 159763 entries, 0 to 180518
Data columns (total 55 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   type                         159763 non-null  object        
 1   days_for_shipping_real       159763 non-null  int64         
 2   days_for_shipment_scheduled  159763 non-null  int64         
 3   benefit_per_order            159763 non-null  float64       
 4   sales_per_customer           159763 non-null  float64       
 5   delivery_status              159763 non-null  object        
 6   late_delivery_risk           159763 non-null  int64         
 7   category_id                  159763 non-null  int64         
 8   category_name                159763 non-null  object        
 9   customer_city                159763 non-null  object        
 10  customer_country             159763 non-null  obje